# 🇮🇱 Israeli Cities RAG — On-Prem with Milvus
### LangChain + Milvus (local) + OpenAI

This notebook is identical in purpose to `main.ipynb` but replaces **Pinecone** (cloud SaaS) with **Milvus Lite** — a fully open-source, on-premises vector database that runs entirely on your machine with zero infrastructure.

**Why Milvus on-prem?**
- No cloud dependency, no API key, no data leaves your machine
- Milvus Lite stores everything in a single local `.db` file
- Drop-in upgrade path: swap the connection URI for a full Milvus server (Docker/K8s) when you need to scale

**Modes supported by this notebook:**
| Mode | URI | When to use |
|------|-----|-------------|
| **Milvus Lite** *(default)* | `./milvus_israeli_cities.db` | Local dev, single machine |
| **Milvus Standalone** | `http://localhost:19530` | On-prem server via Docker |
| **Milvus Cluster** | `http://<host>:19530` | Production on-prem cluster |

## Step 1: Install Dependencies

> ⚠️ **Windows users:** `milvus-lite` (the embedded `.db` file mode) is **not available on Windows**. You must run Milvus via Docker instead. Run the cell below once to start the container, then proceed normally.


In [1]:
# ── Start Milvus Standalone via Docker Compose (Windows) ─────────────────────
# Run this cell ONCE. Requires Docker Desktop to be installed AND running.
# Uses docker-compose.yml (3 containers: etcd + minio + milvus) — the officially
# supported setup. More reliable than the single-container embedded-etcd approach.
import subprocess, os

compose_dir = os.path.dirname(os.path.abspath("__file__"))   # notebook directory

result = subprocess.run(
    ["docker", "compose", "up", "-d"],
    capture_output=True, text=True,
    cwd=compose_dir,
)

combined = result.stderr + result.stdout

if result.returncode == 0:
    print("✅ Milvus stack started (etcd + minio + milvus).")
    print("   Available at http://localhost:19530")
    print("   ⏳ Wait ~30 seconds for all services to become healthy before running Step 6.")
elif "pipe/dockerDesktopLinuxEngine" in combined or "_ping" in combined:
    print("❌ Docker Desktop is not running.")
    print("   → Start Docker Desktop from the Start Menu or system tray, then re-run this cell.")
elif "docker: command not found" in combined or "not recognized" in combined:
    print("❌ Docker is not installed or not on PATH.")
    print("   → Install Docker Desktop from https://www.docker.com/products/docker-desktop/")
else:
    # Print raw output for debugging
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)


✅ Milvus stack started (etcd + minio + milvus).
   Available at http://localhost:19530
   ⏳ Wait ~30 seconds for all services to become healthy before running Step 6.


In [2]:
# pymilvus[model] includes Milvus Lite (no server needed)
# langchain-milvus provides the LangChain vector store integration
! pip install -q pymilvus langchain-milvus langchain langchain-openai langchain-community openai python-dotenv requests pandas tiktoken langchain-text-splitters langchain-classic

## Step 2: Imports & Configuration

In [3]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_milvus import Milvus
from langchain_core.documents import Document
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

load_dotenv(".env")

# Uncomment the lines below ONLY if you are on the Intel corporate VPN/proxy
# and OpenAI calls are failing. Clearing these allows direct internet access.
# for _var in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy", "NO_PROXY", "no_proxy"):
#     os.environ.pop(_var, None)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ── Milvus connection ────────────────────────────────────────────────────────
# milvus-lite (.db file) is Linux/macOS only — not available on Windows.
# Uses docker-compose.yml (etcd + minio + milvus). Start with the Docker setup cell above.
# MILVUS_URI = "./milvus_israeli_cities.db"   # Milvus Lite (Linux/macOS only)
MILVUS_URI  = "http://localhost:19530"          # Milvus Standalone via Docker Compose
# MILVUS_URI = "http://<host>:19530"            # Milvus Cluster (on-prem)
MILVUS_TOKEN = ""                               # leave empty for local; set for secured servers
COLLECTION_NAME = "israeli_cities"

print("✅ Configuration loaded")
print(f"   OpenAI key: {'✅ set' if OPENAI_API_KEY else '❌ missing — set OPENAI_API_KEY in .env'}")
print(f"   Milvus URI: {MILVUS_URI}")


✅ Configuration loaded
   OpenAI key: ✅ set
   Milvus URI: http://localhost:19530


## Step 3: Fetch Wikipedia Articles on Israeli Cities
Same data source as the Pinecone notebook — rich Wikipedia article text about Israeli cities.

In [4]:
ISRAELI_CITIES = [
    "Tel Aviv", "Jerusalem", "Haifa", "Rishon LeZion", "Petah Tikva",
    "Ashdod", "Netanya", "Beer Sheva", "Bnei Brak", "Holon",
    "Bat Yam", "Rehovot", "Ashkelon", "Beit Shemesh", "Kfar Saba",
    "Herzliya", "Ra'anana", "Nazareth", "Modi'in-Maccabim-Re'ut", "Acre",
    "Eilat", "Tiberias", "Safed", "Nahariya", "Lod", "Ramla",
    "Givatayim", "Ramat Gan", "Or Yehuda", "Ness Ziona",
    "Kiryat Ata", "Kiryat Gat", "Dimona", "Arad", "Mitzpe Ramon",
    "Kiryat Shmona", "Afula", "Hadera", "Netivot", "Sderot",
]

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS  = {"User-Agent": "IsraeliCitiesRAG/1.0 (educational project; contact@example.com)"}

def fetch_wikipedia_article(title: str) -> dict | None:
    params = {
        "action": "query", "prop": "extracts",
        "exintro": True, "explaintext": True,
        "redirects": True, "titles": title, "format": "json",
    }
    try:
        resp = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        pages = resp.json()["query"]["pages"]
        page  = next(iter(pages.values()))
        if "missing" in page or not page.get("extract", "").strip():
            return None
        return {"title": page["title"], "text": page["extract"].strip()}
    except Exception as e:
        print(f"  ⚠️  Could not fetch '{title}': {e}")
        return None

articles = []
print(f"Fetching {len(ISRAELI_CITIES)} Wikipedia articles...\n")
for city in ISRAELI_CITIES:
    art = fetch_wikipedia_article(city)
    if art:
        articles.append(art)
        print(f"  ✅ {art['title']}  ({len(art['text'])} chars)")
    else:
        print(f"  ⚠️  Skipped: {city}")

print(f"\n✅ Fetched {len(articles)} articles")
print(f"Average length: {sum(len(a['text']) for a in articles) // len(articles)} chars")

Fetching 40 Wikipedia articles...

  ✅ Tel Aviv  (2895 chars)
  ✅ Jerusalem  (4401 chars)
  ✅ Haifa  (2579 chars)
  ✅ Rishon LeZion  (744 chars)
  ✅ Petah Tikva  (707 chars)
  ✅ Ashdod  (1511 chars)
  ✅ Netanya  (923 chars)
  ✅ Beersheba  (2432 chars)
  ✅ Bnei Brak  (400 chars)
  ✅ Holon  (325 chars)
  ✅ Bat Yam  (210 chars)
  ✅ Rehovot  (215 chars)
  ✅ Ashkelon  (1858 chars)
  ✅ Beit Shemesh  (474 chars)
  ✅ Kfar Saba  (316 chars)
  ✅ Herzliya  (623 chars)
  ✅ Ra'anana  (730 chars)
  ✅ Nazareth  (2113 chars)
  ✅ Modi'in-Maccabim-Re'ut  (1082 chars)
  ✅ Acre  (1235 chars)
  ✅ Eilat  (1673 chars)
  ✅ Tiberias  (2238 chars)
  ✅ Safed  (4207 chars)
  ✅ Nahariya  (188 chars)
  ✅ Lod  (2046 chars)
  ✅ Ramla  (1856 chars)
  ✅ Givatayim  (605 chars)
  ✅ Ramat Gan  (484 chars)
  ✅ Or Yehuda  (179 chars)
  ✅ Ness Ziona  (192 chars)
  ✅ Kiryat Ata  (191 chars)
  ✅ Kiryat Gat  (422 chars)
  ✅ Dimona  (424 chars)
  ✅ Arad  (26 chars)
  ✅ Mitzpe Ramon  (327 chars)
  ✅ Kiryat Shmona  (297 chars)
  ✅

## Step 4: Convert to LangChain Documents

In [5]:
raw_docs = [
    Document(
        page_content=article["text"],
        metadata={"source": "wikipedia", "city": article["title"]},
    )
    for article in articles
]

print(f"✅ Created {len(raw_docs)} documents")
print("\n--- Sample document ---")
print(f"City: {raw_docs[0].metadata['city']}")
print(raw_docs[0].page_content[:600])

✅ Created 40 documents

--- Sample document ---
City: Tel Aviv
Tel Aviv, officially Tel Aviv-Yafo, and also known as Tel Aviv-Jaffa, is the most populous city in the Gush Dan metropolitan area of Israel. Located on the Israeli Mediterranean coastline and with a population of 495,230, it is the economic and technological center of the country and a global high-tech hub. If East Jerusalem is considered part of Israel, Tel Aviv is the country's second-most-populous city, after Jerusalem; if not, Tel Aviv is the most populous city, ahead of West Jerusalem.
Tel Aviv is governed by the Tel Aviv-Yafo Municipality, headed by Mayor Ron Huldai, and is home to most 


## Step 5: Split Documents into Chunks

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " "],
)

chunks = splitter.split_documents(raw_docs)
print(f"✅ Split into {len(chunks)} chunks (from {len(raw_docs)} documents)")
print(f"\n--- Sample chunk ---")
print(chunks[0].page_content)

✅ Split into 128 chunks (from 40 documents)

--- Sample chunk ---
Tel Aviv, officially Tel Aviv-Yafo, and also known as Tel Aviv-Jaffa, is the most populous city in the Gush Dan metropolitan area of Israel. Located on the Israeli Mediterranean coastline and with a population of 495,230, it is the economic and technological center of the country and a global high-tech hub. If East Jerusalem is considered part of Israel, Tel Aviv is the country's second-most-populous city, after Jerusalem; if not, Tel Aviv is the most populous city, ahead of West Jerusalem.


## Step 6: Create Milvus Collection & Upload Embeddings
Connects to the Milvus server started in the Docker setup cell and uploads all document chunks.

⚠️ **Only run this once** — re-running will upload duplicates. To start fresh, set `drop_old=True` or run `docker compose down -v` in the notebook directory and restart.


In [7]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY,
)

# Upload all chunks to Milvus (creates the collection automatically)
print(f"Uploading {len(chunks)} chunks to Milvus at '{MILVUS_URI}'...")
vector_store = Milvus.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    connection_args={
        "uri": MILVUS_URI,
        **({"token": MILVUS_TOKEN} if MILVUS_TOKEN else {}),
    },
    drop_old=False,   # set True to wipe the collection and re-upload
)
print("✅ All chunks uploaded to Milvus!")
print(f"   Collection: {COLLECTION_NAME}")
print(f"   Location:   {MILVUS_URI}")

Uploading 128 chunks to Milvus at 'http://localhost:19530'...
✅ All chunks uploaded to Milvus!
   Collection: israeli_cities
   Location:   http://localhost:19530


## Step 7: Build the RAG Chain
Connect the Milvus retriever to an LLM. After the first run you can skip Step 6 and start here directly — the data is already persisted locally.

In [8]:
# Re-connect to the existing local collection (no re-upload needed on subsequent runs)
vector_store = Milvus(
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
    connection_args={
        "uri": MILVUS_URI,
        **({"token": MILVUS_TOKEN} if MILVUS_TOKEN else {}),
    },
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY,
)

# Custom prompt
prompt_template = """You are an expert on Israeli cities, towns, and localities.
Use ONLY the context below (from Wikipedia) to answer the question.
If the answer is not in the context, say "I don't have that information in the dataset."

Context:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"],
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT},
)

print("✅ RAG chain ready!")

✅ RAG chain ready!


## Step 8: Ask Questions!

In [9]:
def ask(question: str, show_sources: bool = False) -> str:
    """Ask a question to the RAG system."""
    result = rag_chain.invoke({"query": question})
    answer = result["result"]

    print(f"❓ {question}")
    print(f"💬 {answer}")

    if show_sources:
        print("\n📄 Sources:")
        for i, doc in enumerate(result["source_documents"], 1):
            city = doc.metadata.get("city", "?")
            print(f"  [{i}] ({city}) {doc.page_content[:200]}...")

    print("-" * 60)
    return answer

# Example questions
ask("What are the largest cities in Israel by population?")
ask("Which localities are in the Northern district?")
ask("Tell me about Tel Aviv")
ask("What is the smallest town in Israel?", show_sources=True)

❓ What are the largest cities in Israel by population?
💬 The largest cities in Israel by population are:

1. Jerusalem (if East Jerusalem is considered part of Israel)
2. Tel Aviv-Yafo (if East Jerusalem is not considered part of Israel)
3. Haifa
4. Rishon LeZion
5. Ashdod
6. Petah Tikva
7. Netanya
------------------------------------------------------------
❓ Which localities are in the Northern district?
💬 The localities in the Northern District of Israel mentioned in the context are Nazareth and Kiryat Shmona.
------------------------------------------------------------
❓ Tell me about Tel Aviv
💬 Tel Aviv, officially known as Tel Aviv-Yafo and also referred to as Tel Aviv-Jaffa, is the most populous city in the Gush Dan metropolitan area of Israel, with a population of 495,230. It is located on the Israeli Mediterranean coastline and serves as the economic and technological center of the country, being recognized as a global high-tech hub. Tel Aviv was given township status within t

"I don't have that information in the dataset."

## Your Turn — Ask Anything!

In [10]:
my_question = "Is Petah Tikva real?"
ask(my_question, show_sources=True)

❓ Is Petah Tikva real?
💬 Yes, Petah Tikva is a real city in Israel.

📄 Sources:
  [1] (Petah Tikva) Petah Tikva (Hebrew: פתח תקווה, pronounced [ˈpetaχ ˈtikva] ), also spelt Petah Tiqwa and known informally as Em HaMoshavot (lit. 'Mother of the Moshavot, or colonies'), is a city in the Central Distri...
  [2] (Petah Tikva) Petah Tikva (Hebrew: פתח תקווה, pronounced [ˈpetaχ ˈtikva] ), also spelt Petah Tiqwa and known informally as Em HaMoshavot (lit. 'Mother of the Moshavot, or colonies'), is a city in the Central Distri...
  [3] (Petah Tikva) In 2023, the city had a population of 267,196, thus being the fourth-largest city in Israel. Its population density is approximately 6,277 inhabitants per square kilometre (16,260/sq mi). Its jurisdic...
  [4] (Petah Tikva) In 2023, the city had a population of 267,196, thus being the fourth-largest city in Israel. Its population density is approximately 6,277 inhabitants per square kilometre (16,260/sq mi). Its jurisdic...
  [5] (Rishon LeZion) Fo

'Yes, Petah Tikva is a real city in Israel.'